In [7]:
from __future__ import annotations

import json
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif, f_classif, VarianceThreshold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

!pip install catboost
from catboost import CatBoostClassifier

from xgboost import XGBClassifier

from pathlib import Path
from google.colab import drive

CV_FOLDS = 5
TREE_SEARCH_ITER = 25
RANDOM_STATE = 142

@dataclass(frozen=True)
class ClassificationTask:
    name: str
    base_column: str
    threshold_type: str
    threshold_value: float | None = None

@dataclass(frozen=True)
class SearchConfig:
    model_name: str
    pipeline: Pipeline
    params: dict[str, list]
    search_kind: str

def _load_dataset():
    drive.mount('/content/drive')
    df = pd.read_excel('/content/drive/MyDrive/МИФИ Машинное обучение/Классическое машинное обучение/Данные_для_курсовой_Классическое_МО.xlsx')
    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])
    return df

def _make_results_subdir(subdir_name):
    base_dir = Path("results")
    base_dir.mkdir(exist_ok=True)
    exp_dir = base_dir / subdir_name
    exp_dir.mkdir(exist_ok=True)
    return exp_dir

def _build_search_configs() -> list['SearchConfig']:
    K_BEST = [15, 25, 50]

    lr_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("vt", VarianceThreshold(threshold=0)),
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=f_classif)),
        ("model", LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, solver='liblinear')),
    ])

    rf_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("vt", VarianceThreshold(threshold=0)),
        ("selector", SelectKBest(score_func=mutual_info_classif)),
        ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
    ])

    xgb_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("vt", VarianceThreshold(threshold=0)),
        ("selector", SelectKBest(score_func=mutual_info_classif)),
        ("model", XGBClassifier(random_state=RANDOM_STATE, use_label_encoder=False, eval_metric='logloss', n_jobs=-1)),
    ])

    knn_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("vt", VarianceThreshold(threshold=0)),
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=f_classif)),
        ("model", KNeighborsClassifier())
    ])

    svc_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("vt", VarianceThreshold(threshold=0)),
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=f_classif)),
        ("model", SVC(random_state=RANDOM_STATE, probability=True))
    ])

    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("vt", VarianceThreshold(threshold=0)),
        ("selector", SelectKBest(score_func=mutual_info_classif)),
        ("model", CatBoostClassifier(random_state=RANDOM_STATE, verbose=False, thread_count=-1))
    ])

    return [
        SearchConfig(
            model_name="LogisticRegression",
            pipeline=lr_pipeline,
            params={
                "selector__k": K_BEST,
                "model__C": [0.01, 0.1, 1.0, 10.0],
                "model__penalty": ['l1', 'l2']
            },
            search_kind="grid"
        ),
        SearchConfig(
            model_name="RandomForest",
            pipeline=rf_pipeline,
            params={
                "selector__k": K_BEST,
                "model__n_estimators": [200, 500],
                "model__max_depth": [None, 10, 20],
                "model__min_samples_leaf": [1, 2, 4],
                "model__max_features": ["sqrt", None]
            },
            search_kind="random"
        ),
        SearchConfig(
            model_name="XGBoost",
            pipeline=xgb_pipeline,
            params={
                "selector__k": K_BEST,
                "model__n_estimators": [100, 300],
                "model__learning_rate": [0.01, 0.1],
                "model__max_depth": [3, 6],
                "model__subsample": [0.8, 1.0]
            },
            search_kind="random"
        ),
        SearchConfig(
            model_name="KNN",
            pipeline=knn_pipeline,
            params={
                "selector__k": [15, 25],
                "model__n_neighbors": [3, 5, 7, 11],
                "model__weights": ['uniform', 'distance']
            },
            search_kind="grid"
        ),
        SearchConfig(
            model_name="SVM",
            pipeline=svc_pipeline,
            params={
                "selector__k": [15, 25],
                "model__C": [0.1, 1, 10],
                "model__kernel": ['rbf', 'linear']
            },
            search_kind="grid"
        ),
        SearchConfig(
            model_name="CatBoost",
            pipeline=cat_pipeline,
            params={
                "selector__k": [15, 25],
                "model__iterations": [100, 300],
                "model__learning_rate": [0.01, 0.05, 0.1],
                "model__depth": [4, 6]
            },
            search_kind="random"
        )
    ]

def _prepare_task_data(task: ClassificationTask) -> tuple[pd.DataFrame, pd.Series]:
    df = _load_dataset().copy()
    target_series = df[task.base_column]

    if task.threshold_type == 'median':
        threshold = target_series.median()
    else:
        threshold = task.threshold_value

    y = (target_series > threshold).astype(int)
    X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])

    return X, y

def _save_confusion_matrix(y_true, y_pred, path: str):
    plt.figure(figsize=(8, 6))
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.savefig(path, dpi=200)
    plt.close()

def run_classification_task(task: ClassificationTask, verbose=False) -> pd.DataFrame:
    if verbose:
        display(f"Start classification task: {task.name}")

    df = _load_dataset().copy()
    X, y = _prepare_task_data(task)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    exp_dir = _make_results_subdir(f"classification_{task.name}")

    records = []
    best_f1 = -np.inf
    best_predictions = None
    best_model_name = ""

    for config in _build_search_configs():
        if verbose:
            display(f"Fit model {config.model_name}")

        if config.search_kind == "grid":
            search = GridSearchCV(config.pipeline, config.params, scoring="f1_weighted", cv=cv, n_jobs=-1)
        else:
            search = RandomizedSearchCV(config.pipeline, config.params, n_iter=TREE_SEARCH_ITER, scoring="f1_weighted", cv=cv, n_jobs=-1, random_state=RANDOM_STATE)

        search.fit(X_train, y_train)
        best_model = search.best_estimator_
        y_pred = best_model.predict(X_test)

        f1 = f1_score(y_test, y_pred, average='weighted')
        record = {
            "task": task.name,
            "model": config.model_name,
            "cv_f1_weighted": float(search.best_score_),
            "test_accuracy": accuracy_score(y_test, y_pred),
            "test_f1_weighted": f1,
            "test_precision": precision_score(y_test, y_pred, average='weighted'),
            "best_params": json.dumps(search.best_params_),
            "metrics": json.dumps(classification_report(y_test, y_pred), ensure_ascii=False),
        }
        records.append(record)
        if verbose:
            display(record)

        if f1 > best_f1:
            best_f1 = f1
            best_predictions = y_pred
            best_model_name = config.model_name

    result_df = pd.DataFrame(records).sort_values(by="test_f1_weighted", ascending=False)
    result_df.to_csv(exp_dir/"comparison.csv", index=False)

    _save_confusion_matrix(y_test, best_predictions, str(exp_dir/"best_confusion_matrix.png"))

    return result_df